<a href="https://colab.research.google.com/github/ced-sys/SubTerra/blob/main/Site_Diff_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rasterio
import rasterio
import rasterio.features
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import scipy.stats as stats

In [ ]:
try:
  import rasterio
  import rasterio.features
  from rasterio.enums import Resampling
  RASTERIO_AVAILABLE=True
except ImportError:
  print("Warning rasterio is not available.")
  RASTERIO_AVAILABLE=False

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
base_path="/content/drive/MyDrive/Company folders"
years=[2023, 2024]
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
embeddings=None
image_features=None
changes_df=None
combined_data=None


In [ ]:
def load_embeddings(base_path, years=[2023, 2024]):
  all_dfs=[]
  base_path=Path(base_path)

  for year in years:
    folder=base_path/ f"{year} embeddings (CSV)"
    if not folder.exists():
      print(f"Warning: Folder {folder} not found")
      continue
    csv_files=list(folder.glob("*.csv"))
    print(f"Found{len(csv_files)} embedding files for {years}")

    for file_path in csv_files:
      try:
        site_id=file_path.stem
        df=pd.read_csv(file_path)

        if df.shape[0]==1:
          df['site_id']=site_id
          df['year']=year
          all_dfs.append(df)
        else:
          numeric_cols=df.select_dtypes(include=[np.number]).columns
          emb_mean=df[numeric_cols].mean().to_frame().T
          emb_mean['site_id']=site_id
          emb['year']=years
          all_dfs.append(emb_mean)

      except Exception as e:
        print(f"Error loading {file_path}: {e}")

    if all_dfs:
      embeddings_df=pd.concat(all_dfs, ignore_index=True)
      print(f"Loaded embeddings: {embeddings_df.shape}")
      return embeddings_df
    else:
      print("No embeddings loaded")
      return None


In [ ]:
def extract_advanced_image_features(tif_path):
  if not RASTERIO_AVAILABLE:
    return None

  try:
    with rasterio.open(tif_path) as src:
      data=src.read()

      features={}

      for band_idx, band in enumerate(data):
        band_name=f"band_{band_idx}"

        features[f'{band_name}_mean']=np.mean(band)
        features[f'{band_name}_std']=np.std(band)
        features[f'{band_name}_median']=np.median(band)
        features[f'{band_name}_min']=np.min(band)
        features[f'{band_name}_max']=np.max(band)
        features[f'{band_name}_range']=np.ptp(band)

        features[f'{band_name}_skew']=stats.skew(band.flatten())
        features[f'{band_name}_kurtosis']=stats.kurtosis(band.flatten())

        features[f'{band_name}_p10']=np.percentile(band, 10)
        features[f'{band_name}_p25']=np.percentile(band, 25)
        features[f'{band_name}_p75']=np.percentile(band, 75)
        features[f'{band_name}_p90']=np.percentile(band, 90)

      if len(data)>1:
        for i in range(len(data)):
          for j in range(i+1, len(data)):
            ratio=np.divide(data[i], data[j], out=np.ones_like(data[i]), where=data[j]!=0)
            features[f'ratio_band{i}_band{j}']=np.mean(ratio)

        flat_bands=[band.flatten() for band in data]
        for i in range(len(flat_bands)):
          for j in range(i+1, len(flat_bands)):
            corr=np.corrcoef(flat_bands[i], flat_bands[j])[0, 1]
            features[f'corr_band{i}_band{j}']=corr if not np.isnan(corr) else 0

      for band_idx, band in enumerate(data):
        gy, gx=np.gradient(band.astype(float))
        grad_mag=np.sqrt(gx**2 + gy**2)
        features[f'band_{band_idx}_gradient_mean']=np.mean(grad_mag)
        features[f'band_{band_idx}_gradient_std']=np.std(grad_mag)

      return features

  except Exception as e:
    print(f"Error extracting features from {tif_path}: {e}")
    return None


In [ ]:
def load_image_features(base_path, years=[2023, 2024]):
  all_features=[]
  base_path=Path(base_path)

  for year in years:
    folder=base_path / f"{year} embeddings (TIF)"
    if not folder.exists():
      print(f"Warning: Folder {folder} not found")
      continue

    tif_files=list(folder.glob("*.tif"))+list(folder.glob("*.TIF"))
    print(f"Found {len(tif_files)} image files for {year}")

    for file_path in tif_files:
      site_id=file_path.stem
      features=extract_advanced_image_features(file_path)

      if features:
        features['site_id']=site_id
        features['year']=years
        all_features.append(features)

  if all_features:
    image_features_df=pd.DataFrame(all_features)
    print(f"Extracted image features: {image_features_df.shape}")
    return image_features_df
  else:
    print("No image features extracted")
    return None

In [ ]:
def compute_temporal_changes(embeddings_df, years=[2023, 2024]):
  if embeddings_df is None:
    print("No embeddings provided!")
    return None
  changes=[]
  embedding_cols=[col for col in embeddings_df.columns if col not in ['site_id', 'year']]

  for site in embeddings_df['site_id'].unique():
    site_data=embeddings_df[embeddings_df['site_id']==site]


    embeddings_by_year={}
    for year in years:
      year_data=site_data[site_data['year']==year]
      if not year_data.empty:
        embeddings_by_year[year]=year_data[embedding_cols].values[0]

    for i in range(len(years)-1):
      year1, year2=years[i], years[i+1]

      if year1 in embeddings_by_year and year2 in embeddings_by_year:
        emb1=embeddings_by_year[year1].reshape(1, -1)
        emb2=embeddings_by_year[year2].reshape(1, -1)

        cos_sim=cosine_similarity(emb1, emb2)[0][0]
        eucl_dist=euclidean_distances(emb1, emb2)[0][0]

        manhattan_dist=np.sum(np.abs(emb1-emb2))

        relative_change=eucl_dist/ (np.linalg.norm(emb1)+1e-10)

        changed.append({
            'site_id':site,
            'year_from':year1,
            'year_to':year2,
            'cosine_similarity':cos_sim,
            'euclidean_distance': eucl_dist,
            'manhattan_distance':manhattan_dist,
            'relative_change':relative_change,
            'cosine_distance':1-cos_sim
        })

  if changes:
    changes_df=pd.DataFrame(changes)
    print(f"Computed {len(changes)} site changes")
    return changes_df
  else:
    print("No changes computed")
    return None

In [ ]:
def combine_datasets(embeddings_df, image_features_df):
  datasets_to_combine=[]

  if embeddings_df is not None:
    emb_pivot=embeddings_df.set_index(['site_id', 'year'])
    emb_cols=[col for col in embeddings_df.columns if not col not in ['site_id', 'year']]
    emb_pivot.columns=[f'emb_{col}' for col in emb_cols]
    datasets_to_combine.append(emb_pivot)

  if image_features_df is not None:
    img_pivot=image_features_df.set_index(['site_id', 'year'])
    img_cols=[col for col in image_features_df.columns if col not in ['site_id', 'year']]
    img_pivot.columns=[f'img_c{col}' for col in img_cols]
    datasets_to_combine.append(img_pivot)

  if datasets_to_combine:
    combined_df=pd.concat(datasets_to_combine, axis=1).reset_index()
    print(f"Combined dataset: {combined_df.shape}")
    return combined_df
  else:
    print("No datasets to combine")
    return None

In [ ]:
def run_clustering_analysis(combined_df, n_clusters=3):
  if combined_df is None:
    print("No combined data available!")
    return None

  feature_cols=[col for col in combined_df.columns if col not in ['site_id', 'year']]
  X=combined_df[feature_cols].fillna(0)

  scaler=StandardScaler()
  X_scaled=scaler.fit_transfrom(X)

  kmeans=KMeans(n_clusters=n_clusters, random_state=42)
  clusters=kmeans.fit_predict(X_scaled)

  result_df=combined_df.copy()
  result_df['cluster']=clusters

  return result_df, X_scaled, scaler

In [ ]:
def run_pca_analysis(combined_df, n_components=None):
  if combined_df is None:
    print("No combined data available!")
    return None

  feature_cols=[col for col in combined_df.columns if col not in ['site_id', 'year']]
  X=combined_df[feature_cols].fillna(0)

  if n_components is None:
    n_components=min(10, X.shape[1])

  scaler=StandardScaler()
  X_scaled=scaler.fit_transform(X)

  pca=PCA(n_components=n_components)
  X_pca=pca.fit_transform(X_scaled)

  pca_cols=[f'PC{i+1}' for i in range(n_components)]
  pca_df=pd.DataFrame(X_pca, columns=pca_cols)
  pca_df['site_id']=combined_df['site_id'].values
  pca_df['year']=combined_df['year'].values

  return pca_df, pca, scaler

In [ ]:
def create_temporal_changes_plots(changes_df):
  if change_df is None:
    print("No changes data available for plotting!")
    return

  fig, axes=plt.subplots(2, 2, figsize=(15, 12))
  fig.suptitle('Temporal Changes Analysis (2023-> 2024)', fontsize=16, fotweight='bold')

  axes[0,0].hist(changes_df['euclidean_distance'], bins=15, alpha=0.7, color='skyblue', edgecolor='black')
  axes[0,0].set_title('Distribution of Euclidean Distances')
  axes[0,0].set_xlabel('Euclidean Distance')
  axes[0,0].set_ylabel('Frequency')

  axes[0,1].hist(changes_df['cosine_similarity'], bins=15, alpha=0.7, color='skyblue', edgecolor='black')
  axes[0,1].set_title('Distribution of Cosine Similarities')
  axes[0,1].set_xlabel('Cosine Similarity')
  axes[0,1].set_ylabel('Frequency')

  axes[1,0].bar(range(len(changes_df)), changes_df['euclidean_distance'], alpha=0.7, color='coral')
  axes[1,0].set_title('Per-Site Euclidean Distance Changes')
  axes[1,0].set_xlabel('Site Index')
  axes[1,0].set_ylabel('Euclidean Distance')
  axes[1,0].tick_params(axis='x', rotation=45)

  axes[1,1].scatter(changes_df['euclidean_distance'], changes_df['cosine_similarity'],
                    alpha=0.7, s=100, color='purple')
  axes[1,1].set_xlabel('Euclidean Distance')
  axes[1,1].set_ylabel('Cosine Distance')
  axes[1,1].set_title('Euclidean vs Cosine Distance')

  plt.tight_layout()
  plt.show()

  print("\nTemporal Changes Summary")
  print(changes_df.describe())

In [ ]:
def create_pca_plots(pca_df, pca, combined_df):
  if pca_df is None or pca is None:
    print("No PCA data available for plotting!")
    return

  fig, axes=plt.subplots(2,2, figsize=(15, 12))
  fig.suptitle('Principal Component Analysis', fontsize=16, fontweight='bold')

  axes[0,0].bar(range(1, len(pca.explained_variance_ratio_)+1),
                pca.explained_variance_ratio_, alpha=0.7, color='gold')
  axes[0,0].set_title('Explained Variance by Principal Component')
  axes[0,0].set_xlabel('Prinicipal Component')
  axes[0,0].set_ylabel('Explained Variance ratio')

  cumsum_var=np.cumsum(pca.explained_variance_ratio_)
  axes[0,1].plot(range(1, len(cumsum_var)+1), cumsum_var, 'o-', color='red', linewidth=2)
  axes[0,1].set_title('Cumulative Explained Variance')
  axes[0,1].set_xlabel('Number of Components')
  axes[0,1].set_ylabel('Cumulative Explained Variance')
  axes[0,1].grid(True, alpha=0.3)

  colors=['red' if year == 2023 else 'blue' for year in pca_df['year']]
  axes[1,0].scatter(pca_df['PC1'], pca_df['PC2'], c=colors, alpha=0.7, s=100)
  axes[1,0].set_xlabel('PC1')
  axes[1,0].set_ylabel('PC2')
  axes[1,0].set_title('PC1 vs PC2 (Red=2023, Blue=2024)')
  axes[1, 0].grid(True, alpha=0.3)

  for site in pca_df['site_id'].unique():
    site_data=pca_df[pca_df['site_id']==site]
    if len(site_data)>1:
      axes[1,1].plot(site_data['PC1'], site_data['PC2'], 'o-', alpha=0.7, linewidth=2)
  axes[1,1].set_xlabel('PC1')
  axes[1,1].set_ylabel('PC2')
  axes[1,1].set_title('Site Trajectories in PC Space')
  axes[1,1].grid(True, alpha=0.3)

  plt.tight_layout()
  plt.show()




In [ ]:
def create_clustering_plots(cluster_df, pca_df=None):
  if cluster_df is None:
    print("No clustering data available!")
    return
  fig, axes=plt.subplots(2, 2, figsize=(15, 12))
  fig.suptitle("Clustering analysis", fontsize=16, fontweight='bold')

  cluster_counts=cluster_df['cluster'].value_counts().sort_index()
  axes[0,0].bar(cluster_counts.index, cluster_counts.values, alpha=0.7, colo='lightcoral')
  axes[0,0].set_title("Clustersize Distribution")
  axes[0,0].set_xlabel('Cluster')
  axes[0,0].set_ylabel('Number of sizes')

  if pca_df is not None:
    pca_df_with_clusters=pca_df.copy()
    pca_df_with_clusters['cluster']=cluster_df['cluster']

    scatter=axes[0,1].scatter(pca_df_with_clusters['PC1'], pca_df_with_clusters['PC2'],
                              c=pca_df_with_clusters['cluster'], cmap='viridis', alpha=0.7, s=100)
    axes[0,1].set_xlabel('PC1')
    axes[0,1].set_ylabel('PC2')
    axes[0,1].set_title('Clusters in PC Space')
    plt.colorbar(scatter, ax=axes[0,1])

  cluster_year=pd.crosstab(cluster_df['cluster'], cluster_df['year'])
  cluster_year.plot(kind='bar', ax=axes[1,0], alpha=0.7)
  axes[1,0].set_title('Cluster Composition by Year')
  axes[1,0].set_xlabel('Cluster')
  axes[1,0].set_ylabel('Count')
  axes[1,0].legend(title='Year')

  axes[1,1].text(0.1, 0.5, 'Cluster Centers Analysis\n(Feature importance visulaization \nwould require additional\nfeature selection methods)',
                 transform=axes[1,1].transAxes, fontsize=12)
  axes[1,1].set_title('Feature Importance (Placeholder)')

  plt.tight_layout()
  plt.show()




In [ ]:
def generate_sumary_report(embeddings_df, image_features_df, changes_df, combined_df):
  if embeddings_df is not None:
    print(f"\nEMBEDDINGS DATA:")
    print(f"- Total Sites: {embeddings_df['site_id'].nunique()} ")
    print(f"- Years covered: {sorted(embeddings_df['year'].unique())}")
    print(f"- Embedding dimensions: {embeddings_df.shape[1]-2}")
    print(f"- Total observations: {len(embeddings_df)}")

  if image_features_df is not None:
    print(f"\nIMAGE FEATURES:")
    print(f"- Total images processed: {len(image_features_df)}")
    print(f"- Years covered: {sorted(embeddings_df['year'].unique())}")
    print(f"- Embeding dimensions: {embeddings_df.shape[1]-2}")
    print(f"- Total Observations: {len(embeddings_df)}")

  if change_features_df is not None:
    print(f"\nTEMPORAL CHANGES")
    print(f" -Site comparisonsL {len(changes_df)}")
    print(f" -Mean Euclidean distance: {changes_df['euclidean_distance'].mean():.4f}")
    print(f" -Mean Cosine Similarity: {changes_df['cosine_similarity'].mean():.4f}")
    print(f"- Most changed site: {changes_df.loc[changes_df['euclidean_distance'].idmax(), 'site_id']}")
    print(f"- Least changed site: {changes_df.loc[changes_df['euclidean_distance'].idmin(), 'site_id']}")

  if combined_df is not None:
    print(f"\nCOMBINED DATASET:")
    print(f"- Total features: {combined_df.shape[1]-2}")
    print(f"- Data Completeness: {(1- combined_df.isnull().sum().sum()/combined_df.size)*100:1f}%")

  print(f"\nAnalysis completed successfully")



In [ ]:
def run_complete_analysis(base_path=base_path, years=[2023, 2024]):
  global embeddings, image_features, changes_df, combined_data

  embeddings=load_embeddings(base_path, years)

  if RASTERIO_AVAILABLE:
    image_features=load_image_features(base_path, years)
  else:
    print("Skipping")
    image_features=None

  changes_df=compute_temporal_changes(embeddings, years)

  combined_data=combine_datasets(embeddings, image_features)

  if changes_df is not None:
    create_temporal_changes_plots(changes_df)

  if combined_data is not None:
    pca_results=run_pca_analysis(combined_data)
    if pca_results is not None:
      pca_df, pca, scaler=pca_results
      create_pca_plots(pca_df, pca, combined_data)


    clustering_results=run_clustering_analysis(combined_data)
    if clustering_results is not None:
      cluster_df, X_scales, cluster_scaler=clustering_results
      create_clustering_plots(cluster_df, pca_df if pca_results else None)

  generate_summary_report(embeddings, image_features, changes_df, combined_data)

  return{
      'embeddings':embeddings,
      'image_features':image_features,
      'changes':changes_df,
      'combined_data':combined_data
  }





In [ ]:
if __name__=="__main__":
  results=run_complete_analysis(base_path=base_path, years=[2023, 2024])

